In [1]:
import tensorflow as tf
import os
import glob,re
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(1,INPUT_SHAPE,37,False)
cnn_model.summary()
#tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
#cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_staticmapping_epoch49_valAcc0.9718_valPrec0.8219_valRecall0.6488.keras')#
cnn_model.load_weights('guitarmidi.keras')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset/training_subset_electric'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
input_filepaths=glob.glob(os.path.join(input_filepaths, '**', '*.tfrecord'), recursive=True)
input_filepaths = sorted(input_filepaths,key=lambda file: int(re.findall('\\d+',os.path.basename(file))[0]) )
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda it,ot: fast_gpu_map(it,ot, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
#converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1777133372.329303   44771 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777133372.357394   44771 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777133374.128387   44771 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Image height:  148
Before string split: (1, 148, 512), max_x=148.0
String slice ranges (in time steps):  [(0, 52), (20, 72), (40, 92), (60, 112), (76, 128), (96, 148)]
String 0: slicing from 0 to 52 (max_x=148.0)
String 1: slicing from 20 to 72 (max_x=148.0)
String 2: slicing from 40 to 92 (max_x=148.0)
String 3: slicing from 60 to 112 (max_x=148.0)
String 4: slicing from 76 to 128 (max_x=148.0)
String 5: slicing from 96 to 148 (max_x=148.0)
After string split:  [(1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64)]


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (1, 148, 256, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (Subtract)          │                   │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (1, 148, 256, 1)  │          0 │ local_contrast[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (1, 148, 64, 8)   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (1, 148, 64, 8)   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (1, 148, 64, 8)   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (1, 148, 64, 8)   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (1, 148, 512)     │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln1      │ (1, 148, 512)     │      1,024 │ reshape_to_1d[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_mha      │ (1, 148, 512)     │    131,776 │ tfm_block1_ln1[0… │
│ (MultiHeadAttentio… │                   │            │ tfm_block1_ln1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_attn_add │ (1, 148, 512)     │          0 │ reshape_to_1d[0]… │
│ (Add)               │                   │            │ tfm_block1_mha[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln2      │ (1, 148, 512)     │      1,024 │ tfm_block1_attn_… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn1     │ (1, 148, 128)     │     65,664 │ tfm_block1_ln2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_drop │ (1, 148, 128)     │          0 │ tfm_block1_ffn1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn2     │ (1, 148, 512)     │     66,048 │ tfm_block1_ffn_d… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_add  │ (1, 148, 512)     │          0 │ tfm_block1_attn_

 Total params: 742,126 (2.83 MB)

...

 Non-trainable params: 3,344 (13.06 KB)

INFO:tensorflow:Assets written to: /tmp/tmpxs8w9omy/assets


INFO:tensorflow:Assets written to: /tmp/tmpxs8w9omy/assets


Saved artifact at '/tmp/tmpxs8w9omy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(1, 37), dtype=tf.float32, name=None)
Captures:
  132978706410832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706410256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706412752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706411216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706413712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706413328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706414864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706413904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706415248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132978706415056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13297870641160

W0000 00:00:1777133380.421197   44771 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1777133380.421257   44771 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1777133380.422666   44771 reader.cc:83] Reading SavedModel from: /tmp/tmpxs8w9omy
I0000 00:00:1777133380.429270   44771 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1777133380.429299   44771 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpxs8w9omy
I0000 00:00:1777133380.500639   44771 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1777133380.521840   44771 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1777133381.145187   44771 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpxs8w9omy
I0000 00:00:1777133381.299773   44771 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 877122 microseconds.
I0000 00:00:1777133381.477716   4477